# Benchmark — Đo hiệu năng 5 thuật toán mật mã (Pure Python)

Imports trực tiếp từ 5 file `.py` thuần Python, **không dùng thư viện crypto**.

| Thuật toán | Thao tác được đo |
|------------|------------------|
| AES-128    | Mã hóa 16B (1 blk), 20B (2 blk), 1KB (64 blk) |
| SHA-256    | Hash 28B (1 chunk), 56B (2 chunks), 1KB (16 chunks) |
| ECIES      | Mã hóa 37B (2× scalar_mul), Giải mã 37B (1× scalar_mul) |
| ECDH       | Sinh khoá, Shared secret, Full handshake, HKDF |
| Schnorr    | Keygen, Ký, Xác minh, Batch verify 5 chữ ký |

In [ ]:
import sys, os, timeit, statistics, secrets

ALGO_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), '.')
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)

# ── SHA-256 ───────────────────────────────────────────────────────────────
from sha256 import (
    sha256,
    pad, message_schedule, compress,
)

# ── AES-128 ───────────────────────────────────────────────────────────────
from aes128_encrypt import (
    aes128_encrypt, aes_encrypt_block,
    key_expansion, pkcs7_pad,
)

# ── ECC shared math (secp256k1) ───────────────────────────────────────────
from ecc_encrypt import (
    G, N, INF,
    mod_inv, point_add, scalar_mul,
    ecies_encrypt, ecies_decrypt,
    kdf as kdf_ecies,
)

# ── ECDH + HKDF ───────────────────────────────────────────────────────────
from ecdh_key_exchange import (
    hmac_sha256, hkdf,
    ecdh_shared_secret, make_keypair,
)

# ── Schnorr (BIP-340) ─────────────────────────────────────────────────────
from schnorr_signature import (
    schnorr_keygen, schnorr_sign, schnorr_verify,
    tagged_hash, hash_challenge, lift_x, point_neg,
)

print('All imports OK.')

All imports OK.


## 0 · Benchmark helper

In [21]:
ALL_RESULTS = []

def bench(label, stmt, setup='pass', repeat=5, number=1):
    times    = timeit.repeat(stmt=stmt, setup=setup, repeat=repeat,
                             number=number, globals=globals())
    times_ms = [t / number * 1000 for t in times]
    result   = {
        'label'  : label,
        'repeat' : repeat,
        'number' : number,
        'min_ms' : min(times_ms),
        'mean_ms': statistics.mean(times_ms),
        'med_ms' : statistics.median(times_ms),
        'max_ms' : max(times_ms),
    }
    print(f"  {label:<58}  "
          f"min={result['min_ms']:8.3f}ms  "
          f"mean={result['mean_ms']:8.3f}ms  "
          f"med={result['med_ms']:8.3f}ms")
    return result

print('Helper ready.')

Helper ready.


## 1 · AES-128

In [22]:
AES_KEY       = bytes.fromhex('7472756F6E67646F76756F6E67313233')  # 'truongdovuong123'
AES_RKEYS     = key_expansion(AES_KEY)
AES_KEY_HEX   = '7472756F6E67646F76756F6E67313233'

PT_16  = b'HELLO_AES128BLK!'          # exactly 16 B
PT_20  = b'HELLO_UTH_UNIVERSITY!'     # 20 B → 2 blocks after PKCS#7
PT_1KB = bytes(range(256)) * 4        # 1024 B → 64 blocks

print('AES-128 data ready.')

AES-128 data ready.


In [23]:
print('=== AES-128 Benchmark ===')

r = bench('AES-128 encrypt 16B  (1 blk, incl. key_expansion)',
          stmt="aes128_encrypt('HELLO_AES128BLK!', AES_KEY)",
          repeat=5, number=20)
ALL_RESULTS.append(r)

r = bench('AES-128 encrypt_block 16B  (1 blk, pre-computed rkeys)',
          stmt='aes_encrypt_block(PT_16, AES_RKEYS)',
          repeat=5, number=20)
ALL_RESULTS.append(r)

r = bench('AES-128 encrypt 20B  (2 blks, incl. key_expansion)',
          stmt="aes128_encrypt('HELLO_UTH_UNIVERSITY!', AES_KEY)",
          repeat=5, number=20)
ALL_RESULTS.append(r)

r = bench('AES-128 encrypt 1KB  (64 blks, incl. key_expansion)',
          stmt='aes128_encrypt(PT_1KB, AES_KEY)',
          repeat=3, number=3)
ALL_RESULTS.append(r)

=== AES-128 Benchmark ===
  AES-128 encrypt 16B  (1 blk, incl. key_expansion)           min=   1.726ms  mean=   1.865ms  med=   1.851ms
  AES-128 encrypt_block 16B  (1 blk, pre-computed rkeys)      min=   1.161ms  mean=   1.276ms  med=   1.244ms
  AES-128 encrypt 20B  (2 blks, incl. key_expansion)          min=   1.846ms  mean=   2.183ms  med=   2.222ms
  AES-128 encrypt 1KB  (64 blks, incl. key_expansion)         min=  55.847ms  mean=  61.976ms  med=  63.387ms


## 2 · SHA-256

In [24]:
MSG_448 = b'abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq'  # NIST 448-bit
PT_1KB  = bytes(range(256)) * 4

assert sha256(b'abc').hex() == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'
print('SHA-256 sanity check passed.')

SHA-256 sanity check passed.


In [25]:
print('=== SHA-256 Benchmark ===')

r = bench('SHA-256 hash 28B  (1 chunk)',
          stmt="sha256(b'HELLO_UTH_UNIVERSITY!!!!!!!')",
          repeat=5, number=50)
ALL_RESULTS.append(r)

r = bench('SHA-256 hash empty string  (1 chunk)',
          stmt="sha256(b'')",
          repeat=5, number=50)
ALL_RESULTS.append(r)

r = bench('SHA-256 hash 56B  (2 chunks — NIST 448-bit vector)',
          stmt='sha256(MSG_448)',
          repeat=5, number=50)
ALL_RESULTS.append(r)

r = bench('SHA-256 hash 1KB  (16 chunks)',
          stmt='sha256(PT_1KB)',
          repeat=5, number=10)
ALL_RESULTS.append(r)

=== SHA-256 Benchmark ===
  SHA-256 hash 28B  (1 chunk)                                 min=   0.290ms  mean=   0.308ms  med=   0.295ms
  SHA-256 hash empty string  (1 chunk)                        min=   0.282ms  mean=   0.322ms  med=   0.315ms
  SHA-256 hash 56B  (2 chunks — NIST 448-bit vector)          min=   0.603ms  mean=   0.632ms  med=   0.637ms
  SHA-256 hash 1KB  (16 chunks)                               min=   5.145ms  mean=   5.323ms  med=   5.294ms


## 3 · ECIES Encrypt / Decrypt

In [26]:
print('Generating ECIES benchmark keypair (takes a moment)...')

ECIES_PRIV  = secrets.randbelow(N - 1) + 1
ECIES_PUB   = scalar_mul(ECIES_PRIV)
ECIES_MSG   = b'Xin chao Bob! Day la tin nhan bi mat.'
ECIES_EPHEM = secrets.randbelow(N - 1) + 1

ECIES_R, ECIES_CT = ecies_encrypt(ECIES_MSG, ECIES_PUB, ECIES_EPHEM)
assert ecies_decrypt(ECIES_R, ECIES_CT, ECIES_PRIV) == ECIES_MSG
print(f'ECIES keypair ready. Message length: {len(ECIES_MSG)} bytes')

Generating ECIES benchmark keypair (takes a moment)...
ECIES keypair ready. Message length: 37 bytes


In [27]:
print('=== ECIES Benchmark ===')

r = bench('ECIES encrypt 37B  (2× scalar_mul)',
          stmt='ecies_encrypt(ECIES_MSG, ECIES_PUB, ECIES_EPHEM)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('ECIES decrypt 37B  (1× scalar_mul)',
          stmt='ecies_decrypt(ECIES_R, ECIES_CT, ECIES_PRIV)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

=== ECIES Benchmark ===
  ECIES encrypt 37B  (2× scalar_mul)                          min= 143.916ms  mean= 152.206ms  med= 151.191ms
  ECIES decrypt 37B  (1× scalar_mul)                          min=  64.978ms  mean=  76.524ms  med=  68.110ms


## 4 · ECDH Key Exchange

In [28]:
print('Generating ECDH benchmark keypairs...')

ECDH_DA, ECDH_QA = make_keypair(secrets.randbelow(N - 1) + 1)
ECDH_DB, ECDH_QB = make_keypair(secrets.randbelow(N - 1) + 1)
ECDH_SALT        = secrets.token_bytes(32)

S_A = ecdh_shared_secret(ECDH_DA, ECDH_QB)
S_B = ecdh_shared_secret(ECDH_DB, ECDH_QA)
assert S_A == S_B
print('ECDH keypairs ready.')

Generating ECDH benchmark keypairs...
ECDH keypairs ready.


In [29]:
print('=== ECDH Benchmark ===')

r = bench('ECDH make_keypair  (1× scalar_mul)',
          stmt='make_keypair(secrets.randbelow(N - 1) + 1)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('ECDH shared_secret  (1× scalar_mul)',
          stmt='ecdh_shared_secret(ECDH_DA, ECDH_QB)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('ECDH full handshake  (4× scalar_mul)',
          stmt=(
              'da, Qa = make_keypair(secrets.randbelow(N-1)+1);'
              'db, Qb = make_keypair(secrets.randbelow(N-1)+1);'
              'ecdh_shared_secret(da, Qb);'
              'ecdh_shared_secret(db, Qa)'
          ),
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('HKDF-SHA256 derive 32B  (pure HMAC, no ECC)',
          stmt="hkdf(S_A[0].to_bytes(32,'big'), 32, ECDH_SALT, b'ECDH-demo')",
          repeat=5, number=500)
ALL_RESULTS.append(r)

=== ECDH Benchmark ===
  ECDH make_keypair  (1× scalar_mul)                          min=  66.547ms  mean=  72.984ms  med=  75.894ms
  ECDH shared_secret  (1× scalar_mul)                         min=  69.120ms  mean=  73.171ms  med=  69.774ms
  ECDH full handshake  (4× scalar_mul)                        min= 266.676ms  mean= 278.309ms  med= 269.059ms
  HKDF-SHA256 derive 32B  (pure HMAC, no ECC)                 min=   2.587ms  mean=   2.636ms  med=   2.658ms


## 5 · Schnorr Signature (BIP-340)

In [30]:
print('Generating Schnorr benchmark keys and signatures...')

SCH_D, SCH_Q, SCH_QX = schnorr_keygen(secrets.randbelow(N - 1) + 1)
SCH_MSG               = b'Transaction: Alice -> Bob, 0.5 BTC'
SCH_NONCE             = secrets.randbelow(N - 1) + 1
SCH_SIG               = schnorr_sign(SCH_MSG, SCH_D, SCH_QX, SCH_NONCE)

assert schnorr_verify(SCH_MSG, SCH_SIG, SCH_QX)

# Prepare 5 independent (msg, sig, pubkey) tuples for batch verify
BATCH_ITEMS = []
for _ in range(5):
    d_i, _, qx_i = schnorr_keygen(secrets.randbelow(N - 1) + 1)
    m_i          = secrets.token_bytes(32)
    k_i          = secrets.randbelow(N - 1) + 1
    sig_i        = schnorr_sign(m_i, d_i, qx_i, k_i)
    BATCH_ITEMS.append((m_i, sig_i, qx_i))

print('Schnorr ready.')

Generating Schnorr benchmark keys and signatures...
Schnorr ready.


In [31]:
def batch_verify(items):
    # Naive sequential verify — no multi-scalar optimisation
    return all(schnorr_verify(msg, sig, pub) for msg, sig, pub in items)

print('=== Schnorr Benchmark ===')

r = bench('Schnorr keygen  (1-2× scalar_mul)',
          stmt='schnorr_keygen(secrets.randbelow(N - 1) + 1)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('Schnorr sign 1 msg  (1× scalar_mul)',
          stmt='schnorr_sign(SCH_MSG, SCH_D, SCH_QX, SCH_NONCE)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('Schnorr verify 1 sig  (2× scalar_mul + lift_x)',
          stmt='schnorr_verify(SCH_MSG, SCH_SIG, SCH_QX)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

r = bench('Schnorr batch_verify 5 sigs  (10× scalar_mul)',
          stmt='batch_verify(BATCH_ITEMS)',
          repeat=3, number=1)
ALL_RESULTS.append(r)

=== Schnorr Benchmark ===
  Schnorr keygen  (1-2× scalar_mul)                           min=  80.431ms  mean= 103.311ms  med=  92.980ms
  Schnorr sign 1 msg  (1× scalar_mul)                         min=  65.944ms  mean=  76.597ms  med=  72.419ms
  Schnorr verify 1 sig  (2× scalar_mul + lift_x)              min= 130.758ms  mean= 142.530ms  med= 146.566ms
  Schnorr batch_verify 5 sigs  (10× scalar_mul)               min= 696.082ms  mean= 722.728ms  med= 734.335ms


## 6 · Summary Table

In [32]:
W = 105
print('=' * W)
print('  BENCHMARK SUMMARY  (Pure Python, no crypto libs)')
print('=' * W)
print(f"{'Algorithm / Operation':<60} {'min':>9} {'mean':>9} {'median':>9} {'max':>9}  unit")
print('-' * W)

sections = [
    ('AES-128', [r for r in ALL_RESULTS if r['label'].startswith('AES')]),
    ('SHA-256', [r for r in ALL_RESULTS if r['label'].startswith('SHA')]),
    ('ECIES',   [r for r in ALL_RESULTS if r['label'].startswith('ECIES')]),
    ('ECDH',    [r for r in ALL_RESULTS if r['label'].startswith('ECDH') or r['label'].startswith('HKDF')]),
    ('Schnorr', [r for r in ALL_RESULTS if r['label'].startswith('Schnorr') or r['label'].startswith('Schnorr batch')]),
]

for section_name, rows in sections:
    print(f'\n  ── {section_name} ──')
    for r in rows:
        val  = r['med_ms']
        unit = 'ms'
        if val < 0.1:
            val *= 1000
            unit = 'µs'
        print(f"  {r['label']:<60}"
              f"  {r['min_ms']:>8.3f}"
              f"  {r['mean_ms']:>8.3f}"
              f"  {r['med_ms']:>8.3f}"
              f"  {r['max_ms']:>8.3f}  ms")

print('\n' + '=' * W)
print('Note: ECC ops dominate due to pure-Python scalar_mul.')
print('      AES & SHA would be <0.001 ms with C/NumPy backends.')

  BENCHMARK SUMMARY  (Pure Python, no crypto libs)
Algorithm / Operation                                              min      mean    median       max  unit
---------------------------------------------------------------------------------------------------------

  ── AES-128 ──
  AES-128 encrypt 16B  (1 blk, incl. key_expansion)                1.726     1.865     1.851     2.007  ms
  AES-128 encrypt_block 16B  (1 blk, pre-computed rkeys)           1.161     1.276     1.244     1.374  ms
  AES-128 encrypt 20B  (2 blks, incl. key_expansion)               1.846     2.183     2.222     2.497  ms
  AES-128 encrypt 1KB  (64 blks, incl. key_expansion)             55.847    61.976    63.387    66.695  ms

  ── SHA-256 ──
  SHA-256 hash 28B  (1 chunk)                                      0.290     0.308     0.295     0.364  ms
  SHA-256 hash empty string  (1 chunk)                             0.282     0.322     0.315     0.395  ms
  SHA-256 hash 56B  (2 chunks — NIST 448-bit vector)        